# Step 2 of 3 — SQL Analysis
**Pipeline:** `01_data_cleaning` → `02_sql_analysis` → `03_eda`

**Reads:** `output/customer_behavior_clean.csv` (from Step 1) and the queries in `queries.sql`
**Writes:** `output/sql_results/*.csv` — one file per query, used by Step 3 (EDA dashboard)

Database settings are in `config.py` (`DB_BACKEND = "mysql"` or `"sqlite"`).

In [ ]:
import pandas as pd
import config

config.require(config.CLEAN_CSV, "01_data_cleaning.ipynb")
df = pd.read_csv(config.CLEAN_CSV, parse_dates=["visit_date"])
print("Loaded clean data:", df.shape)

## 1. Connect to the database

In [ ]:
conn = config.get_connection()

# Connection test
print(pd.read_sql("SELECT 1 AS connected", conn))
print(f"{config.DB_BACKEND.upper()} connected successfully!")

## 2. Upload the cleaned data to a table

In [ ]:
df.to_sql(name=config.TABLE_NAME, con=conn, if_exists="replace", index=False)

rows_in_db = pd.read_sql(f"SELECT COUNT(*) AS n FROM {config.TABLE_NAME}", conn)["n"][0]
print(f"Uploaded {rows_in_db} rows to table '{config.TABLE_NAME}'")
assert rows_in_db == len(df), "Row count in database does not match the CSV!"

## 3. Load the SQL queries from `queries.sql`

In [ ]:
def load_queries(path):
    queries, name, buf = {}, None, []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith("-- name:"):
            if name:
                queries[name] = "\n".join(buf).strip().rstrip(";")
            name, buf = line.split(":", 1)[1].strip(), []
        elif name:
            buf.append(line)
    if name:
        queries[name] = "\n".join(buf).strip().rstrip(";")
    return queries

queries = load_queries(config.QUERIES_SQL)
print("Queries found:", list(queries))

## 4. Run every query and save the results (→ Step 3)

In [ ]:
results = {}
for name, sql in queries.items():
    results[name] = pd.read_sql(sql, conn)
    results[name].to_csv(config.SQL_OUT_DIR / f"{name}.csv", index=False)
    print(f"✔ {name:18s} -> {len(results[name])} rows saved")

### Five KPIs

In [ ]:
print("DASHBOARD KPIs")
display(results["kpis"].T.rename(columns={0: "value"}).round(2))

### All result tables

In [ ]:
for name, table in results.items():
    if name == "kpis":
        continue
    print(f"\n{name.upper().replace('_', ' ')}")
    display(table)

In [ ]:
if config.DB_BACKEND == "sqlite":
    conn.close()
print("✔ Step 2 complete  →  now run 03_eda.ipynb")